<a href="https://colab.research.google.com/github/c-marq/AI-Thinking-CAI1001C/blob/main/11-Natural-Language-Processing/Guided-Project/GP11_NLP_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GP11: NLP & Sentiment Analysis
### CAI1001C: Artificial Intelligence (AI) Thinking | Chapter 11

**Guided Project — Reference Material (Not Graded)**

In this demo, you'll explore how computers process and understand human language — the technology behind autocomplete, translation, Siri, and ChatGPT. You'll run sentiment analysis on real restaurant reviews using two different approaches:

- **Part 1:** VADER — a rule-based sentiment analyzer (fast, simple, transparent)
- **Part 2:** HuggingFace Transformer — a pre-trained deep learning model (slower, smarter, less transparent)

By the end, you'll see where these approaches agree, where they disagree, and why that matters.

---

**Key Concept:** NLP (Natural Language Processing) is the branch of AI that teaches machines to understand human language. Sentiment analysis — determining whether text is positive, negative, or neutral — is one of the most common NLP tasks in industry.

## Learning Objectives

After completing this demo, you will be able to:

1. **Explain** how text is converted into data a computer can process (tokenization, stop words)
2. **Run** sentiment analysis using VADER and interpret the compound score
3. **Compare** rule-based (VADER) vs. ML-based (HuggingFace) sentiment analysis
4. **Identify** where sentiment models fail — sarcasm, negation, informal language
5. **Describe** how language bias affects NLP model accuracy across dialects

## Setup

Run the cell below to install and import everything we need. This only takes a few seconds.

In [ ]:
# ============================================
# Run this cell first — installs and imports
# ============================================

!pip install vaderSentiment -q

import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import re
import warnings
warnings.filterwarnings('ignore')

print("All libraries installed and imported!")
print("  - pandas: data handling")
print("  - vaderSentiment: rule-based sentiment analysis")
print("  - re: text cleaning with regular expressions")

## Load the Dataset

We're using a real dataset of **10,000 restaurant reviews** with star ratings (1-5). This is actual customer feedback — messy, opinionated, and sometimes surprising.

In [ ]:
# ============================================
# Load the restaurant reviews dataset
# ============================================

url = "https://raw.githubusercontent.com/manthanpatel98/Restaurant-Review-Sentiment-Analysis/master/Restaurant%20reviews.csv"
df = pd.read_csv(url)

# Clean: drop missing values and ensure Rating is numeric
df = df.dropna(subset=['Review', 'Rating']).copy()
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce')
df = df.dropna(subset=['Rating']).copy()
df['Rating'] = df['Rating'].astype(int)

print(f"Loaded {len(df)} restaurant reviews")
print(f"\nRating distribution:")
for rating in sorted(df['Rating'].unique()):
    count = len(df[df['Rating'] == rating])
    bar = "█" * (count // 100)
    print(f"  ⭐ {rating}: {count:,} reviews  {bar}")

print(f"\nSample 5-star review:")
print(f'  "{df[df["Rating"]==5].iloc[0]["Review"][:120]}..."')
print(f"\nSample 1-star review:")
print(f'  "{df[df["Rating"]==1].iloc[0]["Review"][:120]}..."')

# Expected Output:
# Loaded 9,954 restaurant reviews
# 1-star: ~1,744 | 2-star: ~703 | 3-star: ~1,239 | 4-star: ~2,442 | 5-star: ~3,826

---

## Part 1A: Text as Data — How Computers Read

Before a computer can analyze sentiment, it needs to break text into pieces it can work with. This process has three steps:

1. **Tokenization** — split text into individual words (tokens)
2. **Stop word removal** — remove common words that don't carry meaning ("the", "was", "is")
3. **Cleaning** — remove punctuation and normalize the text

In [ ]:
# ============================================
# Example 11.1: Text Processing Pipeline
# Purpose: Show how text becomes data
# ============================================

sample_review = "The restaurant was absolutely amazing! Best Cuban food I've ever had."

# Step 1: Tokenize
tokens = sample_review.lower().split()
print(f"Original:  {sample_review}")
print(f"Tokens:    {tokens}")
print(f"Count:     {len(tokens)} words")

# Step 2: Remove stop words
stop_words = {'the', 'was', 'a', 'an', 'is', 'it', 'to', 'and',
              'of', 'in', 'for', 'on', 'at', 'by', 'we', 'our', 'my',
              'this', 'that', 'with', 'but', 'not', 'very', 'so', 'just'}
filtered = [w for w in tokens if w.strip('!.,?\'') not in stop_words]
print(f"\nAfter stop words: {filtered}")

# Step 3: Remove punctuation
clean = [re.sub(r'[^\w\s]', '', w) for w in filtered if re.sub(r'[^\w\s]', '', w)]
print(f"After cleanup:    {clean}")
print(f"\nWe went from {len(tokens)} tokens to {len(clean)} meaningful words")

### Try It Yourself — Text Processing

Change the `sample_review` below to your own sentence and re-run. Try a review in Spanglish, a sarcastic review, or a very short one.

In [ ]:
# ▶ MODIFY THIS: Replace with your own review
sample_review = "The restaurant was absolutely amazing! Best Cuban food I've ever had."

# --- Processing pipeline ---
tokens = sample_review.lower().split()
filtered = [w for w in tokens if w.strip('!.,?\'') not in stop_words]
clean = [re.sub(r'[^\w\s]', '', w) for w in filtered if re.sub(r'[^\w\s]', '', w)]
print(f"Original ({len(tokens)} words): {sample_review}")
print(f"Cleaned  ({len(clean)} words):  {clean}")

---

## Part 1B: VADER Sentiment Analysis

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) is a rule-based sentiment analyzer. It has a dictionary of ~7,500 words with preset sentiment scores:

- "Excellent" = very positive, "terrible" = very negative, "okay" = slightly positive
- It also uses rules for punctuation (!), capitalization (AMAZING), and negation (not good)

VADER returns a **compound score** from -1 (most negative) to +1 (most positive):
- Compound ≥ 0.05 → **POSITIVE**
- Compound ≤ -0.05 → **NEGATIVE**
- Between → **NEUTRAL**

In [ ]:
# ============================================
# Example 11.2: VADER on Individual Reviews
# ============================================

analyzer = SentimentIntensityAnalyzer()

test_reviews = [
    "The food was absolutely delicious! Best meal I've had in months.",
    "Terrible service. We waited over an hour and the food was cold.",
    "The restaurant was okay. Nothing special but not bad either.",
    "I can't believe how bad this place has gotten. Used to be great.",
    "Not the worst meal I've ever had, but definitely not worth the price.",
]

print("VADER Sentiment Analysis — Individual Reviews")
print("=" * 65)

for i, review in enumerate(test_reviews, 1):
    scores = analyzer.polarity_scores(review)
    label = "POSITIVE" if scores['compound'] >= 0.05 else (
        "NEGATIVE" if scores['compound'] <= -0.05 else "NEUTRAL")
    print(f"\nReview {i}: \"{review}\"")
    print(f"  Pos: {scores['pos']:.3f} | Neg: {scores['neg']:.3f} | Neu: {scores['neu']:.3f}")
    print(f"  Compound: {scores['compound']:.4f} → {label}")

# Expected Output:
# Review 1: 0.8678 → POSITIVE  (correct)
# Review 2: -0.4767 → NEGATIVE (correct)
# Review 3: 0.5568 → POSITIVE  (debatable — "okay" is really neutral)
# Review 4: 0.7876 → POSITIVE  (WRONG — this is negative!)
# Review 5: 0.5716 → POSITIVE  (WRONG — "not worth the price" is negative!)

### Did You Catch That?

**Reviews 4 and 5** — VADER got them completely wrong:

- **Review 4:** *"I can't believe how bad this place has gotten. Used to be great."* → VADER says POSITIVE (0.79). It sees "great" and scores it positively, but misses that "used to be" means it's NOT great anymore.

- **Review 5:** *"Not the worst meal... but definitely not worth the price."* → VADER says POSITIVE (0.57). The double negation confuses the rule-based system.

**This is the core limitation of rule-based NLP:** VADER counts words but doesn't understand sentences. It can't handle sarcasm, complex negation, or context that changes meaning.

### Try It Yourself — Break VADER

Type your own reviews below and try to *break* VADER — find sentences where the score doesn't match the actual sentiment.

In [ ]:
# ▶ MODIFY THESE: Add your own reviews
my_reviews = [
    "This place is absolutely wonderful!",
    "I would not recommend this to anyone.",
    "Yeah right, like this was the best food ever...",  # Try sarcasm!
]

for review in my_reviews:
    scores = analyzer.polarity_scores(review)
    label = "POSITIVE" if scores['compound'] >= 0.05 else (
        "NEGATIVE" if scores['compound'] <= -0.05 else "NEUTRAL")
    print(f"\"{review}\"")
    print(f"  → Compound: {scores['compound']:.4f} = {label}\n")

---

## Part 1C: VADER on 10,000 Reviews

Now let's run VADER on the entire dataset and compare its sentiment scores to the actual star ratings.

In [ ]:
# ============================================
# Example 11.3: VADER on Full Dataset
# ============================================

# Score every review
df['vader_compound'] = df['Review'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound']
)

# Average VADER score by star rating
print("Average VADER Score by Star Rating")
print("=" * 45)
for rating in sorted(df['Rating'].unique()):
    subset = df[df['Rating'] == rating]
    avg = subset['vader_compound'].mean()
    bar = "█" * int((avg + 1) * 15)
    print(f"  ⭐ {rating}: {avg:>+.4f}  (n={len(subset):,})  {bar}")

# Binary accuracy: 4-5 stars = positive, 1-2 stars = negative
df_binary = df[df['Rating'].isin([1, 2, 4, 5])].copy()
df_binary['actual_pos'] = (df_binary['Rating'] >= 4).astype(int)
df_binary['vader_pos'] = (df_binary['vader_compound'] >= 0.05).astype(int)

correct = (df_binary['vader_pos'] == df_binary['actual_pos']).sum()
total = len(df_binary)
print(f"\nVADER Accuracy (4-5★=positive, 1-2★=negative, 3★ excluded)")
print(f"  {correct:,}/{total:,} = {correct/total*100:.1f}%")

fp = len(df_binary[(df_binary['vader_pos']==1) & (df_binary['actual_pos']==0)])
fn = len(df_binary[(df_binary['vader_pos']==0) & (df_binary['actual_pos']==1)])
print(f"\n  False positives (VADER=pos, actual 1-2★): {fp}")
print(f"  False negatives (VADER=neg, actual 4-5★): {fn}")
print(f"  VADER is {fp//fn}x more likely to miss negative reviews!")

# Expected Output:
# 1★ avg: -0.34, 2★: -0.06, 3★: +0.44, 4★: +0.77, 5★: +0.77
# Accuracy: ~89%  |  ~594 false positives vs ~74 false negatives

### Try It Yourself — Change the Threshold

What happens when you raise or lower the compound score cutoff?

In [ ]:
# ▶ MODIFY THIS VALUE: Try 0.0, 0.1, 0.2, 0.3
threshold = 0.05

df_binary['vader_pred'] = (df_binary['vader_compound'] >= threshold).astype(int)
correct = (df_binary['vader_pred'] == df_binary['actual_pos']).sum()
accuracy = correct / len(df_binary) * 100
fp = len(df_binary[(df_binary['vader_pred']==1) & (df_binary['actual_pos']==0)])
fn = len(df_binary[(df_binary['vader_pred']==0) & (df_binary['actual_pos']==1)])

print(f"Threshold: {threshold}")
print(f"Accuracy:  {accuracy:.1f}%")
print(f"False positives: {fp}  |  False negatives: {fn}")

---

## Part 1D: The Language Bias Test

The most important experiment in this demo. We test whether VADER treats all language styles equally.

**5 pairs of reviews** — each pair expresses the **same sentiment**, but one uses formal English and the other uses Spanglish/slang. In Miami, bilingual communication is everyday life. Does the model handle it fairly?

In [ ]:
# ============================================
# The Language Bias Test
# ============================================

bias_pairs = [
    ("The food was excellent and the service was outstanding.",
     "La comida was fire bro, the service was on point no cap."),
    ("I was very disappointed with my meal. It was overcooked and bland.",
     "Nah that food was nasty fr, todo overcooked y no sabia a nada."),
    ("A wonderful dining experience. Highly recommended.",
     "Bro this spot is lowkey amazing, you gotta try it asap."),
    ("The portions were too small for the price they charge.",
     "Dale bro they giving you poquito food for too much money smh."),
    ("The atmosphere was lovely and the staff were very attentive.",
     "The vibes were chill af and the staff was super nice tho."),
]

print("Formal English vs. Spanglish/Informal — Same Sentiment")
print("=" * 55)
print(f"  {'Pair':<5} {'Formal':>8} {'Informal':>9} {'Gap':>8} {'Same Label?':>12}")
print("-" * 50)

formal_scores = []
informal_scores = []
for i, (formal, informal) in enumerate(bias_pairs, 1):
    f_s = analyzer.polarity_scores(formal)['compound']
    i_s = analyzer.polarity_scores(informal)['compound']
    f_l = "POS" if f_s >= 0.05 else ("NEG" if f_s <= -0.05 else "NEU")
    i_l = "POS" if i_s >= 0.05 else ("NEG" if i_s <= -0.05 else "NEU")
    match = "✅" if f_l == i_l else "❌ NO"
    formal_scores.append(f_s)
    informal_scores.append(i_s)
    print(f"  {i:<5} {f_s:>+.4f} {i_s:>+.4f} {f_s-i_s:>+.4f} {match:>12}")

print(f"\n  Average formal:   {np.mean(formal_scores):.4f}")
print(f"  Average informal: {np.mean(informal_scores):.4f}")
print(f"  Average gap:      {np.mean(formal_scores)-np.mean(informal_scores):+.4f}")

# Expected: Pair 1 gets OPPOSITE labels (POS vs NEG)
# Average gap ~+0.30 — formal English consistently scores higher

### The Bias Is Real

**Pair 1:** Same positive sentiment, completely opposite classification:
- Formal: *"excellent and outstanding"* → **POSITIVE (+0.83)**
- Spanglish: *"fire bro, on point no cap"* → **NEGATIVE (-0.56)**

VADER doesn't know "fire" means excellent, "on point" means outstanding, or "no cap" means "I'm serious." It reads "no" as negation.

The average gap of **+0.30** means formal English consistently scores higher than Spanglish — even when expressing the same sentiment. In Miami, a business using this tool would systematically undervalue bilingual customer feedback.

### Try It Yourself — Your Own Bias Pairs

In [ ]:
# ▶ MODIFY THESE: Write your own matched pair
formal_review = "The dessert was exceptional and beautifully presented."
informal_review = "Bro that dessert was bussin, looked crazy good too."

f_s = analyzer.polarity_scores(formal_review)['compound']
i_s = analyzer.polarity_scores(informal_review)['compound']
f_l = "POS" if f_s >= 0.05 else ("NEG" if f_s <= -0.05 else "NEU")
i_l = "POS" if i_s >= 0.05 else ("NEG" if i_s <= -0.05 else "NEU")

print(f"Formal:   \"{formal_review}\"  → {f_s:+.4f} [{f_l}]")
print(f"Informal: \"{informal_review}\"  → {i_s:+.4f} [{i_l}]")
print(f"Gap: {f_s-i_s:+.4f}")
if f_l != i_l:
    print("⚠️ DIFFERENT LABELS — same sentiment, different classification!")

---

## Part 2: HuggingFace Transformer — The "Smart" Approach

VADER uses a dictionary lookup. What if we used a model that *learned* language from millions of examples?

The model we'll use (`distilbert`) was trained on thousands of movie reviews and learned to understand context, negation, and sentence structure.

- **VADER:** looks up words → adds scores → done
- **Transformer:** reads the whole sentence → understands word relationships → classifies

⚠️ This cell downloads a ~250MB model. May take 1-2 minutes.

In [ ]:
# ============================================
# Load HuggingFace Sentiment Model
# ============================================

!pip install transformers -q

from transformers import pipeline

print("Downloading model (~250MB, may take 1-2 minutes)...")
hf_analyzer = pipeline("sentiment-analysis",
                       model="distilbert-base-uncased-finetuned-sst-2-english",
                       device=-1)
print("✅ Model loaded!")

### Head-to-Head: VADER vs. HuggingFace

Same 5 reviews, two different approaches:

In [ ]:
# ============================================
# VADER vs. HuggingFace — Same Reviews
# ============================================

test_reviews = [
    "The food was absolutely delicious! Best meal I've had in months.",
    "Terrible service. We waited over an hour and the food was cold.",
    "The restaurant was okay. Nothing special but not bad either.",
    "I can't believe how bad this place has gotten. Used to be great.",
    "Not the worst meal I've ever had, but definitely not worth the price.",
]

print("Head-to-Head: VADER vs. HuggingFace")
print("=" * 75)

for i, review in enumerate(test_reviews, 1):
    v = analyzer.polarity_scores(review)['compound']
    v_label = "POS" if v >= 0.05 else ("NEG" if v <= -0.05 else "NEU")
    hf = hf_analyzer(review[:512])[0]
    match = "✅" if v_label == hf['label'][:3] else "❌"
    print(f"\n  Review {i}: \"{review[:55]}...\"")
    print(f"    VADER: {v:+.4f} [{v_label}]  |  HF: {hf['label']} ({hf['score']:.1%})  {match}")

# Expected: Reviews 1-3 agree. Reviews 4-5: VADER=POS, HF=NEGATIVE
# HuggingFace catches the negation/context that VADER misses!

### Does HuggingFace Fix the Language Bias?

In [ ]:
# ============================================
# HuggingFace on the Language Bias Pairs
# ============================================

print("Language Bias: VADER vs. HuggingFace")
print("=" * 60)
print(f"  {'Pair':<5} {'VADER-Formal':>13} {'VADER-Informal':>15} {'HF-Formal':>10} {'HF-Inform':>10}")
print("-" * 60)

for i, (formal, informal) in enumerate(bias_pairs, 1):
    v_f = "POS" if analyzer.polarity_scores(formal)['compound'] >= 0.05 else "NEG"
    v_i = "POS" if analyzer.polarity_scores(informal)['compound'] >= 0.05 else "NEG"
    hf_f = hf_analyzer(formal[:512])[0]['label'][:3]
    hf_i = hf_analyzer(informal[:512])[0]['label'][:3]
    print(f"  {i:<5} {v_f:>11}  {v_i:>14}  {hf_f:>9}  {hf_i:>9}")

# Expected: Pair 1 — BOTH VADER and HF fail on informal version!
# The transformer makes the same mistake as the dictionary.

### The Surprising Result

**Both models fail on the same Spanglish pair.** The transformer — trained on millions of examples — makes the same mistake as simple dictionary-based VADER.

Why? Both were trained primarily on Standard English. Neither has seen enough Spanglish or informal language to understand "fire" (excellent), "no cap" (seriously), or "on point" (outstanding).

This isn't a bug — it's a **systemic training data gap**. The model doesn't know what it doesn't know.

### Accuracy Comparison

In [ ]:
# ============================================
# Accuracy comparison on 100-review sample
# ============================================

df_binary = df[df['Rating'].isin([1, 2, 4, 5])].copy()
df_binary['actual_pos'] = (df_binary['Rating'] >= 4).astype(int)
df_binary['vader_compound'] = df_binary['Review'].apply(
    lambda x: analyzer.polarity_scores(str(x))['compound'])
df_binary['vader_pred'] = (df_binary['vader_compound'] >= 0.05).astype(int)

np.random.seed(42)
idx = np.random.choice(len(df_binary), 100, replace=False)
sample = df_binary.iloc[idx].copy()

print("Running HuggingFace on 100 random reviews...")
hf_preds = [1 if hf_analyzer(str(t)[:512])[0]['label']=='POSITIVE' else 0
            for t in sample['Review']]
sample = sample.copy()
sample['hf_pred'] = hf_preds

v_acc = (sample['vader_pred'] == sample['actual_pos']).sum()
h_acc = (sample['hf_pred'] == sample['actual_pos']).sum()

print(f"\n{'Method':<30} {'Correct':>8} {'Accuracy':>10}")
print("-" * 50)
print(f"  {'VADER (rule-based)':<28} {v_acc}/100 {v_acc:.1f}%")
print(f"  {'HuggingFace (transformer)':<28} {h_acc}/100 {h_acc:.1f}%")
print(f"\n  Difference: {h_acc-v_acc:+.0f} percentage points")

# Expected: VADER ~91%, HuggingFace ~93%, difference ~+2 pts

---

## Summary: What We Learned

| | VADER (Rule-Based) | HuggingFace (Transformer) |
|---|---|---|
| **How it works** | Dictionary lookup + rules | Pre-trained on millions of examples |
| **Speed** | Instant | Slower (needs model download) |
| **Accuracy on standard English** | ~89% | ~93% |
| **Handles negation/context** | Poorly (missed Reviews 4 & 5) | Well (caught both) |
| **Handles Spanglish/slang** | Failed on Pair 1 | Also failed on Pair 1 |
| **Transparency** | You can see exactly why | Black box |

**The bottom line:** The transformer is smarter at understanding sentence structure, but both models share the same blind spot — language that doesn't look like their training data.

---

### What's Next

**Group Lab:** Explore NLP models on HuggingFace.co — test sentiment, translation, and run your own bias experiments in your browser.

**Homework:** Go deeper — test multiple models, analyze patterns, and write about what this means for real businesses and communities in Miami.